# 淘宝用户行为数据分析 - 数据清洗

## 1. 加载数据

本Notebook用于对原始数据进行清洗和预处理，为后续分析做准备。

In [1]:
import sys
sys.path.append('../src')

from data_loader import load_raw_item_data,load_raw_user_data,init_sqlite_db
from config import CLEANED_DATA_FILE

user_df = load_raw_user_data()
print(f'加载成功，共读取到{len(user_df)}条用户数据')
item_df = load_raw_item_data()
print(f'加载成功，共读取到{len(item_df)}条商品数据')

加载成功，共读取到12256906条用户数据
加载成功，共读取到480723条商品数据


## 2.处理重复数据

In [2]:
print('商品数据')
print(f'重复数据共计{item_df.duplicated().sum()}条')

print('\n')

print('用户数据')
print(f'重复数据共计{user_df.duplicated().sum()}条')
user_df = user_df.drop_duplicates()

print(f'清洗后重复数据共计{user_df.duplicated().sum()}条')

商品数据
重复数据共计0条


用户数据
重复数据共计4092866条
清洗后重复数据共计0条


## 3.处理缺失值

In [3]:
print('商品数据')
print(f'各列缺失值数量：\n{item_df.isnull().sum()}')
item_df['item_geohash'] = item_df['item_geohash'].fillna('unknown')
print(f'清洗后各列缺失值数量：\n{item_df.isnull().sum()}')

print('\n')

print('用户数据')
print(f'各列缺失值数量：\n{user_df.isnull().sum()}')
user_df['user_geohash'].fillna('unknown', inplace=True)
print(f'清洗后各列缺失值数量：\n{user_df.isnull().sum()}')

商品数据
各列缺失值数量：
item_id               0
item_geohash     307063
item_category         0
dtype: int64
清洗后各列缺失值数量：
item_id          0
item_geohash     0
item_category    0
dtype: int64


用户数据
各列缺失值数量：
user_id                 0
item_id                 0
behavior_type           0
user_geohash      4308015
item_category           0
time                    0
date                    0
hour                    0
behavior_label          0
dtype: int64
清洗后各列缺失值数量：
user_id           0
item_id           0
behavior_type     0
user_geohash      0
item_category     0
time              0
date              0
hour              0
behavior_label    0
dtype: int64


## 4.验证行为类型

In [4]:
print(f'行为类型不在[1，2，3，4]之中的数量：{len(user_df['behavior_type']) - user_df['behavior_type'].isin([1, 2, 3, 4]).sum()}')

行为类型不在[1，2，3，4]之中的数量：0


## 5.数据融合

In [5]:
item_df.drop(['item_category'],axis=1,inplace=True)
df = user_df.merge(item_df, on='item_id', how='left',suffixes=('_user', '_item'))
df['item_geohash'] = df['item_geohash'].fillna('unknown')
print('数据融合成功')

数据融合成功


## 6.保存数据

In [6]:
df.to_csv(CLEANED_DATA_FILE, index=False)
init_sqlite_db(df)
print('数据保存成功')

数据保存成功
